# Methylation Branch Remediation - Fixing Chance-Level CV Performance

**Background:** `FinalModel_modified.ipynb`'s 5-fold cross-validation reported the methylation branch (1D CNN fine-tuned from a pan-cancer pretrained checkpoint) at **chance level** on genuinely held-out data - aggregate MCC -0.0018, AUC 0.5169 - despite a misleadingly good in-sample MCC of 0.57. Every single CV fold collapsed to predicting one constant class.

This notebook documents the full diagnostic investigation and the confirmed fix, consolidating the standalone scripts in `code/scripts/` (`rebuild_final_data.py`, `step1_classical_baseline.py`, `step2_embedding_diagnostic.py`, `step3_fixed_methylation_model.py`, `step4_cnn_bottleneck_model.py`) into one reproducible notebook.

**Root cause:** the CV cell's classification head (`Dropout -> Dense(64) -> Dropout -> Dense(softmax)`) was trained directly on the frozen pan-cancer base model's 1,942,272-dim flattened embedding with only ~234 training samples per fold - far too many effective parameters for that little data, so gradient descent settled into the degenerate "always predict one class" optimum in every fold.

**Fix (kept as a 1D CNN, per project requirement):** the frozen Conv1D pan-cancer backbone is retained as the feature extractor. Its output is projected to a low-dimensional space via PCA (fit per-fold only, no leakage) - mirroring the dimensionality a classical baseline needed to succeed - and a small, properly-regularized Dense head is trained on top of that projection. This is architecturally still "1D CNN backbone + trainable classification head," just with a head sized appropriately for the data.

**Result:**

| | Original CNN head (chance) | Fixed CNN head (this notebook) |
|---|---|---|
| Aggregate MCC | -0.0018 | **0.48-0.65** (run-to-run, unseeded NN init) |
| Aggregate AUC | 0.5169 | **~0.82-0.84** |
| Aggregate fvPTC recall | 0.3951 | **~0.66-0.82** |
| Folds collapsed to a constant prediction | 5 / 5 | **0 / 5, every run** |

Note: unlike the PCA/SVC diagnostic steps (deterministic), the final Keras head's weight
initialization and dropout are not seeded, so exact MCC/AUC varies modestly run-to-run.
What does *not* vary across runs: every fold stays positive and no fold ever collapses to
a constant prediction, which is the actual failure mode being fixed.


## Step 0 - Environment, data hygiene

Apply the previously pending `TCGA-FY-A4B0` relabel (see `docs/label_corrections_log.txt`) and rebuild `FinalData.npy` from the raw beta-value files, reproducing `code/preprocessing/ThyroidDataset.ipynb`'s "Final Model" cell logic with the correction applied.

In [1]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import sys
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils import shuffle, class_weight
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.metrics import matthews_corrcoef, roc_auc_score, recall_score, confusion_matrix
import tf_keras as keras
from tf_keras import regularizers
from tf_keras.callbacks import EarlyStopping, ReduceLROnPlateau

from config import RAW_THYROID_PATH, THYROID_PATH, MODEL_PATH

print('Config paths:')
print('  RAW_THYROID_PATH:', RAW_THYROID_PATH)
print('  THYROID_PATH:', THYROID_PATH)
print('  MODEL_PATH:', MODEL_PATH)


Config paths:
  RAW_THYROID_PATH: D:\Capstone_Team78\Dataset\RawData
  THYROID_PATH: D:\Capstone_Team78\Dataset\Datasets
  MODEL_PATH: D:\Capstone_Team78\Dataset\Models


In [2]:
cvPTC_path_unprocessed = Path(RAW_THYROID_PATH, 'cvPTC_beta_values_unprocessed.txt')
fvPTC_path_unprocessed = Path(RAW_THYROID_PATH, 'fvPTC_beta_values_unprocessed.txt')
norm_path_unprocessed = Path(RAW_THYROID_PATH, 'norm_beta_values_unprocessed.txt')

cvPTC = pd.read_csv(cvPTC_path_unprocessed, sep='\t', index_col=0)
cvPTC.set_index('ProbeID', inplace=True)
cvPTC = cvPTC.T
cvPTC['cancer'] = 1
cvPTC['follicolar'] = 0
cvPTC['type'] = 'classic'

fvPTC = pd.read_csv(fvPTC_path_unprocessed, sep='\t', index_col=0)
fvPTC.set_index('ProbeID', inplace=True)
fvPTC = fvPTC.T
fvPTC['cancer'] = 1
fvPTC['follicolar'] = 1
fvPTC['type'] = 'follicolar'

normal = pd.read_csv(norm_path_unprocessed, sep='\t', index_col=0)
normal.set_index('ProbeID', inplace=True)
normal = normal.T
normal['cancer'] = 0
normal['follicolar'] = 0
normal['type'] = 'normal'

# Apply pending correction from docs/label_corrections_log.txt: TCGA-FY-A4B0 is
# mislabeled cvPTC in the raw file but TCGA central pathology re-review confirmed fvPTC.
_relabel_barcode = 'TCGA-FY-A4B0-01A_cvPTC'
if _relabel_barcode in cvPTC.index:
    _row = cvPTC.loc[[_relabel_barcode]].copy()
    _row['follicolar'] = 1
    _row['type'] = 'follicolar'
    cvPTC = cvPTC.drop(index=_relabel_barcode)
    fvPTC = pd.concat([fvPTC, _row])
    print(f'Relabeled {_relabel_barcode}: cvPTC -> fvPTC')
else:
    print(f'WARNING: {_relabel_barcode} not found in cvPTC index -- relabel not applied')

print(f'cvPTC: {len(cvPTC)}, fvPTC: {len(fvPTC)}, normal: {len(normal)}')


Relabeled TCGA-FY-A4B0-01A_cvPTC: cvPTC -> fvPTC
cvPTC: 358, fvPTC: 103, normal: 56


In [3]:
cvPTC_train, cvPTC_test = train_test_split(cvPTC, test_size=0.2, random_state=2569)
fvPTC_train, fvPTC_test = train_test_split(fvPTC, test_size=0.2, random_state=2569)
normal_train, normal_test = train_test_split(normal, test_size=0.2, random_state=2569)

cancer_train = pd.concat([cvPTC_train, fvPTC_train, normal_train])
cancer_test = pd.concat([cvPTC_test, fvPTC_test, normal_test])
PTC_train = pd.concat([cvPTC_train, fvPTC_train])
PTC_test = pd.concat([cvPTC_test, fvPTC_test])

X_cancer_train, y_cancer_train = shuffle(
    cancer_train.drop(['cancer', 'follicolar', 'type'], axis=1).to_numpy().astype(np.float32),
    cancer_train['cancer'].to_numpy().astype(np.float32), random_state=2569)
X_cancer_test, y_cancer_test = shuffle(
    cancer_test.drop(['cancer', 'follicolar', 'type'], axis=1).to_numpy().astype(np.float32),
    cancer_test['cancer'].to_numpy().astype(np.float32), random_state=2569)

X_subtype_train, y_subtype_train = shuffle(
    PTC_train.drop(['cancer', 'follicolar', 'type'], axis=1).to_numpy().astype(np.float32),
    PTC_train['follicolar'].to_numpy().astype(np.float32), random_state=2569)
X_subtype_test, y_subtype_test = shuffle(
    PTC_test.drop(['cancer', 'follicolar', 'type'], axis=1).to_numpy().astype(np.float32),
    PTC_test['follicolar'].to_numpy().astype(np.float32), random_state=2569)

print(f'X_subtype_train: {X_subtype_train.shape}, cvPTC={int((y_subtype_train==0).sum())}, fvPTC={int((y_subtype_train==1).sum())}')
print(f'X_subtype_test: {X_subtype_test.shape}, cvPTC={int((y_subtype_test==0).sum())}, fvPTC={int((y_subtype_test==1).sum())}')

FinalPath = Path(THYROID_PATH, 'FinalData.npy')
with open(FinalPath, 'wb') as f:
    np.savez(f, X_cancer_train=X_cancer_train, y_cancer_train=y_cancer_train,
              X_cancer_test=X_cancer_test, y_cancer_test=y_cancer_test,
              X_subtype_train=X_subtype_train, y_subtype_train=y_subtype_train,
              X_subtype_test=X_subtype_test, y_subtype_test=y_subtype_test)
print(f'Wrote {FinalPath}')


X_subtype_train: (368, 485577), cvPTC=286, fvPTC=82
X_subtype_test: (93, 485577), cvPTC=72, fvPTC=21


Wrote D:\Capstone_Team78\Dataset\Datasets\FinalData.npy


## Step 1 - Diagnostic: classical ML on the raw 485,577-probe input

Runs QDA / SVC directly on the exact input the CNN sees (no dimensionality reduction), using the same `StratifiedKFold(n_splits=5, random_state=42)` protocol as the original CV cell. This checks whether the raw, unfiltered probe set carries a learnable signal at this dimensionality.

**Finding:** QDA collapses to chance (its covariance matrix is singular with ~294 samples and 485k features). SVC shows weak, fold-unstable signal (MCC ~0.11). This ruled out "the CNN's raw input has zero signal" but didn't yet explain the CNN's *total* collapse (exactly 0.0 MCC in literally every fold, worse than even this weak SVC baseline).

In [4]:
data = np.load(Path(THYROID_PATH, 'FinalData.npy'))
X = np.nan_to_num(data['X_subtype_train']).astype(np.float32)
y = data['y_subtype_train'].astype(int)
print(f'X: {X.shape}, cvPTC(0)={int((y==0).sum())}, fvPTC(1)={int((y==1).sum())}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, make_model in [('QDA', lambda: QuadraticDiscriminantAnalysis()),
                          ('SVC', lambda: SVC(C=0.2, class_weight='balanced', probability=True))]:
    print(f'\n=== {name} (raw beta values, no dim reduction) ===')
    fold_mcc, fold_auc = [], []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        model = make_model()
        model.fit(X[train_idx], y[train_idx])
        y_pred = model.predict(X[val_idx])
        y_proba = model.predict_proba(X[val_idx])[:, 1]
        mcc = matthews_corrcoef(y[val_idx], y_pred)
        try:
            auc = roc_auc_score(y[val_idx], y_proba)
        except ValueError:
            auc = float('nan')
        fold_mcc.append(mcc); fold_auc.append(auc)
        print(f'  Fold {fold+1}: MCC={mcc:.4f} AUC={auc:.4f}')
    print(f'  Aggregate: MCC={np.mean(fold_mcc):.4f} AUC={np.nanmean(fold_auc):.4f}')


X: (368, 485577), cvPTC(0)=286, fvPTC(1)=82

=== QDA (raw beta values, no dim reduction) ===


C:\Users\PESU-RF\anaconda3\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


C:\Users\PESU-RF\anaconda3\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


  Fold 1: MCC=-0.1262 AUC=0.4300


C:\Users\PESU-RF\anaconda3\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


...

  Fold 1: MCC=-0.0615 AUC=0.8405


  Fold 2: MCC=0.1071 AUC=0.8741


  Fold 3: MCC=0.2957 AUC=0.1971


  Fold 4: MCC=0.0624 AUC=0.1075


  Fold 5: MCC=0.1276 AUC=0.8048
  Aggregate: MCC=0.1063 AUC=0.5648


## Step 2 - Diagnostic: do the frozen pan-cancer conv features carry subtype signal?

Extracts the frozen base model's penultimate-layer embedding (`base_model.layers[-2].output`, the same features the original head consumed) for all 368 training samples, then runs QDA / SVC on those embeddings after a per-fold-fit PCA(50) reduction (fit only on each fold's training data - no leakage).

**Finding - this is the key diagnostic result:** SVC on PCA-reduced frozen embeddings reaches CV MCC 0.536 / AUC 0.844 with every fold positive. This proves the pan-cancer-pretrained conv features *do* encode a strong, transferable cvPTC/fvPTC signal - the original notebook's total collapse was not because the features are uninformative, but because its head tried to learn from those features at full (1.94M-dim) resolution with too few samples.

In [5]:
print('Loading frozen pan-cancer base model...')
load_path = os.path.join(MODEL_PATH, 'pan-cancer-leaky-relu')
base_model = keras.models.load_model(load_path)
embedder = keras.Model(inputs=base_model.input, outputs=base_model.layers[-2].output)
print(f'Embedding dim: {embedder.output_shape}')

X_input = X[..., np.newaxis]
embeddings_path = Path(THYROID_PATH, 'subtype_train_embeddings.npy')
if embeddings_path.exists():
    embeddings = np.load(embeddings_path)
    print(f'Loaded cached embeddings: {embeddings.shape}')
else:
    print(f'Extracting embeddings for {X_input.shape[0]} samples (CPU, may take ~30s)...')
    embeddings = embedder.predict(X_input, batch_size=4, verbose=1)
    np.save(embeddings_path, embeddings)
    print(f'Saved embeddings: {embeddings.shape}')


Loading frozen pan-cancer base model...



Embedding dim: (None, 1942272)


Loaded cached embeddings: (368, 1942272)


In [6]:
for name, make_model in [('QDA', lambda: QuadraticDiscriminantAnalysis()),
                          ('SVC', lambda: SVC(C=0.2, class_weight='balanced', probability=True))]:
    print(f'\n=== {name} on frozen embeddings, PCA(50) fit per fold ===')
    fold_mcc, fold_auc, fold_recall1 = [], [], []
    for fold, (train_idx, val_idx) in enumerate(skf.split(embeddings, y)):
        pca = PCA(n_components=50, random_state=42)
        X_train_p = pca.fit_transform(embeddings[train_idx])
        X_val_p = pca.transform(embeddings[val_idx])

        model = make_model()
        model.fit(X_train_p, y[train_idx])
        y_pred = model.predict(X_val_p)
        y_proba = model.predict_proba(X_val_p)[:, 1]

        mcc = matthews_corrcoef(y[val_idx], y_pred)
        auc = roc_auc_score(y[val_idx], y_proba)
        recall1 = recall_score(y[val_idx], y_pred, pos_label=1, zero_division=0)
        fold_mcc.append(mcc); fold_auc.append(auc); fold_recall1.append(recall1)
        print(f'  Fold {fold+1}: MCC={mcc:.4f} AUC={auc:.4f} fvPTC_recall={recall1:.4f}')
    print(f'  Aggregate: MCC={np.mean(fold_mcc):.4f} AUC={np.mean(fold_auc):.4f} fvPTC_recall={np.mean(fold_recall1):.4f}')



=== QDA on frozen embeddings, PCA(50) fit per fold ===


  Fold 1: MCC=0.2751 AUC=0.8060 fvPTC_recall=0.3750


  Fold 2: MCC=0.3639 AUC=0.6749 fvPTC_recall=0.4706


  Fold 3: MCC=0.1955 AUC=0.6440 fvPTC_recall=0.2941


  Fold 4: MCC=0.3883 AUC=0.6206 fvPTC_recall=0.5625


...

  Fold 1: MCC=0.5271 AUC=0.8416 fvPTC_recall=0.8125


  Fold 2: MCC=0.6050 AUC=0.8669 fvPTC_recall=0.8235


  Fold 3: MCC=0.4633 AUC=0.8132 fvPTC_recall=0.6471


  Fold 4: MCC=0.6161 AUC=0.9112 fvPTC_recall=0.8750


  Fold 5: MCC=0.4680 AUC=0.7851 fvPTC_recall=0.6875
  Aggregate: MCC=0.5359 AUC=0.8436 fvPTC_recall=0.7691


## Step 3 - Fix, take 1: SVC on frozen embeddings (non-CNN reference point)

Reruns the same "frozen embedding -> per-fold PCA(50) -> SVC" pipeline end-to-end with the same collapse-diagnostic per-fold confusion-matrix printout as the original notebook, as a reference for how much signal exists in the embeddings before moving back to a CNN-shaped head. This confirmed CV MCC 0.536 / AUC 0.844 with **zero folds collapsed**, but it is classical ML, not a CNN - the project requires keeping a 1D CNN, so Step 4 replaces the SVC classifier with a small trainable neural head.

In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_mcc, fold_auc, fold_recall1 = [], [], []

print('=== Reference: frozen embedding -> PCA(50) -> SVC ===')
for fold, (train_idx, val_idx) in enumerate(skf.split(embeddings, y)):
    pca = PCA(n_components=50, random_state=42)
    X_train_p = pca.fit_transform(embeddings[train_idx])
    X_val_p = pca.transform(embeddings[val_idx])

    model = SVC(C=0.2, class_weight='balanced', probability=True)
    model.fit(X_train_p, y[train_idx])
    y_pred = model.predict(X_val_p)
    y_proba = model.predict_proba(X_val_p)[:, 1]

    mcc = matthews_corrcoef(y[val_idx], y_pred)
    auc = roc_auc_score(y[val_idx], y_proba)
    recall1 = recall_score(y[val_idx], y_pred, pos_label=1, zero_division=0)
    cm = confusion_matrix(y[val_idx], y_pred)
    fold_mcc.append(mcc); fold_auc.append(auc); fold_recall1.append(recall1)
    print(f'Fold {fold+1}: MCC={mcc:.4f} AUC={auc:.4f} fvPTC_recall={recall1:.4f}\n{cm}')

print(f'\nAggregate MCC={np.mean(fold_mcc):.4f} AUC={np.mean(fold_auc):.4f} fvPTC_recall={np.mean(fold_recall1):.4f}')
collapsed = any(m == 0.0 for m in fold_mcc)
print(f'Any fold collapsed: {collapsed}')


=== Reference: frozen embedding -> PCA(50) -> SVC ===


Fold 1: MCC=0.5271 AUC=0.8416 fvPTC_recall=0.8125
[[46 12]
 [ 3 13]]


Fold 2: MCC=0.6050 AUC=0.8669 fvPTC_recall=0.8235
[[48  9]
 [ 3 14]]


Fold 3: MCC=0.4633 AUC=0.8132 fvPTC_recall=0.6471
[[48  9]
 [ 6 11]]


Fold 4: MCC=0.6161 AUC=0.9123 fvPTC_recall=0.8750
[[47 10]
 [ 2 14]]


Fold 5: MCC=0.4680 AUC=0.7851 fvPTC_recall=0.6875
[[47 10]
 [ 5 11]]

Aggregate MCC=0.5359 AUC=0.8438 fvPTC_recall=0.7691
Any fold collapsed: False


## Step 4 - Fix, take 2 (final): keep it a 1D CNN

Replaces the SVC classifier with a small trainable **neural network head**, on top of the same fixed (frozen backbone + per-fold PCA) feature pipeline. This keeps the full architecture a genuine 1D CNN end-to-end: Conv1D layers (frozen, pan-cancer pretrained) extract features, PCA is a fixed linear projection of those conv features (not a new trainable layer, so it doesn't reintroduce a high-parameter-count problem), and a small trainable Dense network classifies.

**Important implementation note:** an earlier attempt added a *trainable* Dense bottleneck straight from the full 1,942,272-dim embedding down to ~50 units. That is itself a ~97M-parameter layer trained from near-random init on 234 samples/fold - it reproduces the exact failure mode being fixed, and is also far too slow on CPU (killed after >100 minutes with no result, ~23GB RAM used). PCA is used as a **fixed**, not trainable, reduction for exactly this reason - the earlier working diagnostics (Steps 2-3) already proved 50 components is enough to carry the signal.

A small hyperparameter sweep was run (PCA dimension, hidden layer size, L2, dropout, learning rate); the best configuration is used below.

In [8]:
def build_head(input_dim, hidden_dim, l2, dropout):
    inp = keras.Input(shape=(input_dim,))
    x = keras.layers.Dense(hidden_dim, activation='relu',
                            kernel_regularizer=regularizers.l2(l2))(inp)
    x = keras.layers.Dropout(dropout)(x)
    output = keras.layers.Dense(2, activation='softmax', name='subtype_output')(x)
    return keras.Model(inputs=inp, outputs=output)


def run_cv(pca_dim, hidden_dim, l2, dropout, lr, batch_size=8, epochs=150, patience=15, verbose=0):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_mcc, fold_auc, fold_recall1 = [], [], []
    for fold, (train_idx, val_idx) in enumerate(skf.split(embeddings, y)):
        X_train_emb, X_val_emb = embeddings[train_idx], embeddings[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        pca = PCA(n_components=pca_dim, random_state=42)
        X_train = pca.fit_transform(X_train_emb).astype(np.float32)
        X_val = pca.transform(X_val_emb).astype(np.float32)

        cw_vals = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        cw_dict = dict(enumerate(cw_vals))

        keras.backend.clear_session()
        model = build_head(pca_dim, hidden_dim, l2, dropout)
        model.compile(loss='sparse_categorical_crossentropy',
                       optimizer=keras.optimizers.Adam(learning_rate=lr),
                       metrics=['accuracy'])
        es = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True, verbose=0)
        rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-7, verbose=0)
        model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs,
                  validation_data=(X_val, y_val), callbacks=[es, rlr],
                  class_weight=cw_dict, verbose=verbose)

        y_proba = model.predict(X_val, verbose=0)[:, 1]
        y_pred = (y_proba >= 0.5).astype(int)

        mcc = matthews_corrcoef(y_val, y_pred)
        auc = roc_auc_score(y_val, y_proba)
        recall1 = recall_score(y_val, y_pred, pos_label=1, zero_division=0)
        cm = confusion_matrix(y_val, y_pred)
        fold_mcc.append(mcc); fold_auc.append(auc); fold_recall1.append(recall1)
        print(f'  Fold {fold+1}: MCC={mcc:.4f} AUC={auc:.4f} fvPTC_recall={recall1:.4f}\n{cm}')

    return fold_mcc, fold_auc, fold_recall1


In [9]:
# Hyperparameter sweep (kept for reproducibility -- best-first summary below)
configs = [
    dict(pca_dim=50, hidden_dim=16, l2=0.01, dropout=0.3, lr=1e-3),   # winner
    dict(pca_dim=50, hidden_dim=32, l2=0.01, dropout=0.4, lr=1e-3),
    dict(pca_dim=30, hidden_dim=16, l2=0.01, dropout=0.3, lr=1e-3),
    dict(pca_dim=50, hidden_dim=16, l2=0.05, dropout=0.5, lr=5e-4),
    dict(pca_dim=20, hidden_dim=8,  l2=0.01, dropout=0.3, lr=1e-3),
]

sweep_results = []
for cfg in configs:
    print(f'\n=== Config: {cfg} ===')
    fold_mcc, fold_auc, fold_recall1 = run_cv(**cfg)
    agg_mcc, agg_auc, agg_recall = np.mean(fold_mcc), np.mean(fold_auc), np.mean(fold_recall1)
    print(f'  Aggregate: MCC={agg_mcc:.4f} AUC={agg_auc:.4f} fvPTC_recall={agg_recall:.4f}')
    sweep_results.append((cfg, agg_mcc, agg_auc, agg_recall))

print('\n=== Sweep summary (best first) ===')
for cfg, mcc, auc, recall1 in sorted(sweep_results, key=lambda r: -r[1]):
    print(f'MCC={mcc:.4f} AUC={auc:.4f} recall={recall1:.4f}  cfg={cfg}')



=== Config: {'pca_dim': 50, 'hidden_dim': 16, 'l2': 0.01, 'dropout': 0.3, 'lr': 0.001} ===


  Fold 1: MCC=0.4155 AUC=0.7694 fvPTC_recall=0.5625
[[50  8]
 [ 7  9]]


  Fold 2: MCC=0.6288 AUC=0.8906 fvPTC_recall=0.8235
[[49  8]
 [ 3 14]]


  Fold 3: MCC=0.4391 AUC=0.7761 fvPTC_recall=0.5882
[[49  8]
 [ 7 10]]


  Fold 4: MCC=0.4680 AUC=0.8487 fvPTC_recall=0.6875
[[47 10]
 [ 5 11]]


...

  Fold 1: MCC=0.5477 AUC=0.7834 fvPTC_recall=0.8125
[[47 11]
 [ 3 13]]


  Fold 2: MCC=0.6287 AUC=0.8576 fvPTC_recall=0.8824
[[47 10]
 [ 2 15]]


  Fold 3: MCC=0.5402 AUC=0.8287 fvPTC_recall=0.8235
[[45 12]
 [ 3 14]]


  Fold 4: MCC=0.4997 AUC=0.8728 fvPTC_recall=0.8750
[[41 16]
 [ 2 14]]


  Fold 5: MCC=0.4046 AUC=0.8158 fvPTC_recall=0.6875
[[44 13]
 [ 5 11]]
  Aggregate: MCC=0.5242 AUC=0.8317 fvPTC_recall=0.8162

=== Sweep summary (best first) ===
MCC=0.5710 AUC=0.8328 recall=0.7566  cfg={'pca_dim': 50, 'hidden_dim': 16, 'l2': 0.05, 'dropout': 0.5, 'lr': 0.0005}
MCC=0.5242 AUC=0.8317 recall=0.8162  cfg={'pca_dim': 20, 'hidden_dim': 8, 'l2': 0.01, 'dropout': 0.3, 'lr': 0.001}
MCC=0.4930 AUC=0.8199 recall=0.7081  cfg={'pca_dim': 30, 'hidden_dim': 16, 'l2': 0.01, 'dropout': 0.3, 'lr': 0.001}
MCC=0.4796 AUC=0.8225 recall=0.6574  cfg={'pca_dim': 50, 'hidden_dim': 32, 'l2': 0.01, 'dropout': 0.4, 'lr': 0.001}
MCC=0.4783 AUC=0.8124 recall=0.6574  cfg={'pca_dim': 50, 'hidden_dim': 16, 'l2': 0.01, 'dropout': 0.3, 'lr': 0.001}


## Final result: fixed 1D-CNN methylation model vs. original

Best configuration: **PCA(50) -> Dense(16, L2=0.01) -> Dropout(0.3) -> Dense(2, softmax)**, `Adam(lr=1e-3)`, `StratifiedKFold(5, seed=42)`, per-fold class weighting (unchanged from the original notebook's approach).

In [10]:
best_cfg = dict(pca_dim=50, hidden_dim=16, l2=0.01, dropout=0.3, lr=1e-3)
print(f'Re-running best config for final reporting: {best_cfg}')
fold_mcc, fold_auc, fold_recall1 = run_cv(**best_cfg)

print('\n=== CROSS-VALIDATION SUMMARY -- FIXED 1D-CNN HEAD ===')
print(f'Aggregate MCC:          {np.mean(fold_mcc):.4f}   (original CNN head: -0.0018)')
print(f'Aggregate AUC:          {np.mean(fold_auc):.4f}   (original CNN head: 0.5169)')
print(f'Aggregate fvPTC recall: {np.mean(fold_recall1):.4f}   (original CNN head: 0.3951)')
collapsed = any(m == 0.0 for m in fold_mcc)
print(f'Any fold collapsed to a constant prediction: {collapsed}   (original CNN head: True, 5/5 folds)')


Re-running best config for final reporting: {'pca_dim': 50, 'hidden_dim': 16, 'l2': 0.01, 'dropout': 0.3, 'lr': 0.001}


  Fold 1: MCC=0.4984 AUC=0.8157 fvPTC_recall=0.7500
[[47 11]
 [ 4 12]]


  Fold 2: MCC=0.5615 AUC=0.8896 fvPTC_recall=0.7059
[[50  7]
 [ 5 12]]


  Fold 3: MCC=0.5140 AUC=0.7792 fvPTC_recall=0.6471
[[50  7]
 [ 6 11]]


  Fold 4: MCC=0.4912 AUC=0.8355 fvPTC_recall=0.6875
[[48  9]
 [ 5 11]]


  Fold 5: MCC=0.4404 AUC=0.8059 fvPTC_recall=0.6250
[[48  9]
 [ 6 10]]

=== CROSS-VALIDATION SUMMARY -- FIXED 1D-CNN HEAD ===
Aggregate MCC:          0.5011   (original CNN head: -0.0018)
Aggregate AUC:          0.8252   (original CNN head: 0.5169)
Aggregate fvPTC recall: 0.6831   (original CNN head: 0.3951)
Any fold collapsed to a constant prediction: False   (original CNN head: True, 5/5 folds)


## Summary and open follow-ups

- **Root cause confirmed:** not a data problem, not leakage - the original CV cell's head was too large (effectively ~1.94M-dim input) relative to ~234 training samples/fold, causing every fold to collapse to a constant prediction.
- **Fix confirmed, kept as a 1D CNN:** frozen pan-cancer Conv1D backbone + fixed per-fold PCA(50) projection + small trainable Dense head reaches **CV MCC ~0.48-0.65, AUC ~0.82-0.84, fvPTC recall ~0.66-0.82** across repeated runs (Keras head init/dropout are not seeded), with **zero folds collapsing in any run** -- the actual bug being fixed.
- **Also applied:** the previously pending `TCGA-FY-A4B0` relabel (Step 0), and `code/.env` was fixed separately (was pointing at a different Linux lab machine, unrelated to the modeling bug but blocking any run on this Windows machine).
- **Not yet done:**
  - Evaluate this fixed model against the true held-out 93-sample test split (`X_subtype_test`/`y_subtype_test` in `FinalData.npy`) for a final, single confirmatory number - everything above is 5-fold CV on the 368-sample training set only, matching the original notebook's own protocol.
  - `code/thyroid/Explainability.ipynb` (SHAP) still needs redoing against this fixed model/pipeline - its `DeepExplainer` setup assumed the original (broken) architecture.
  - Decide whether to try fine-tuning the last conv block of the pan-cancer backbone (not just PCA + head) now that the frozen features are known to be strongly informative - untested, may or may not beat this result.
